In [2]:
# !pip uninstall isanlp -y && pip install git+https://github.com/iinemo/isanlp.git
# !pip install isanlp_rst

In [3]:
import re
import json
from difflib import SequenceMatcher
from typing import List, Dict, Tuple
import pandas as pd
from tqdm import tqdm

In [4]:
df = pd.read_csv("labeled.csv")[['text', 'label']]
print(df.iloc[0].to_dict())

{'text': 'Я стала вновь жить, радоваться, с великолепным аттестатом окончила школу, поступила туда, куда хотела, ходила в церковь.', 'label': '[{"start":0,"end":120,"text":"Я стала вновь жить, радоваться, с великолепным аттестатом окончила школу, поступила туда, куда хотела, ходила в церковь.","labels":["простое предложение"]}]'}


## Метрики

In [8]:
def _normalize(text: str) -> str:
    return re.sub(r'[^\wа-яёa-z0-9]', '', text.lower())


def _similarity(s1: str, s2: str) -> float:
    return SequenceMatcher(None, s1, s2).ratio()


def parse_gold_labels(gold_label_str: str) -> List[Dict]:
    try:
        return json.loads(gold_label_str)
    except Exception:
        return []


def clauses_to_label_format(clauses: List[Tuple[int, int, str]], label_type: str = "простое предложение") -> str:
    labels_list = [
        {"start": start, "end": end, "text": text, "labels": [label_type]}
        for start, end, text in clauses
    ]
    return json.dumps(labels_list, ensure_ascii=False)


def _greedy_match_with_positions(pred_clauses: List[Tuple[int, int, str]], gold_clauses: List[Dict], threshold: float = 0.85) -> int:
    if not pred_clauses or not gold_clauses:
        return 0

    matched_pred_indices: set[int] = set()
    tp = 0

    for gold in gold_clauses:
        g_start = gold['start']
        g_end = gold['end']
        g_norm = _normalize(gold.get('text', ''))

        best_score = 0.0
        best_idx = -1

        for i, (p_start, p_end, p_text) in enumerate(pred_clauses):
            if i in matched_pred_indices:
                continue

            p_norm = _normalize(p_text)
            text_sim = _similarity(g_norm, p_norm)

            overlap_start = max(g_start, p_start)
            overlap_end = min(g_end, p_end)
            overlap = max(0, overlap_end - overlap_start)
            union_length = max(g_end - g_start, p_end - p_start)
            pos_iou = overlap / union_length if union_length > 0 else 0.0

            combined_score = 0.7 * text_sim + 0.3 * pos_iou

            if combined_score > best_score:
                best_score = combined_score
                best_idx = i

        if best_score >= threshold and best_idx != -1:
            tp += 1
            matched_pred_indices.add(best_idx)

    return tp


def compute_metrics_and_predictions(text: str, gold_label_str: str, predicted_clauses: List[Tuple[int, int, str]], threshold: float = 0.85) -> Dict:
    gold_clauses = parse_gold_labels(gold_label_str)
    parser_json = clauses_to_label_format(predicted_clauses)

    if not gold_clauses:
        return {
            "parser": parser_json, "precision": 0.0, "recall": 0.0, "f1": 0.0,
            "jaccard_exact": 0.0, "jaccard_relaxed": 0.0,
            "num_pred": len(predicted_clauses), "num_gold": 0, "error": "No gold labels"
        }

    tp = _greedy_match_with_positions(predicted_clauses, gold_clauses, threshold)

    prec = tp / len(predicted_clauses) if predicted_clauses else 0.0
    rec = tp / len(gold_clauses) if gold_clauses else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

    p_set = {(_normalize(t), s, e) for s, e, t in predicted_clauses}
    g_set = {(_normalize(g['text']), g['start'], g['end']) for g in gold_clauses}

    if not p_set and not g_set:
        jaccard_exact = 1.0
    else:
        intersection = len(p_set & g_set)
        union = len(p_set | g_set)
        jaccard_exact = intersection / union if union > 0 else 0.0

    denom = len(predicted_clauses) + len(gold_clauses) - tp
    jaccard_relaxed = tp / denom if denom > 0 else 1.0

    return {
        "parser": parser_json, "precision": prec, "recall": rec, "f1": f1,
        "jaccard_exact": jaccard_exact, "jaccard_relaxed": jaccard_relaxed,
        "num_pred": len(predicted_clauses), "num_gold": len(gold_clauses), "error": None
    }


def print_summary_stats(result_df: pd.DataFrame) -> None:
    print("\n" + "=" * 60)
    print("SUMMARY STATISTICS")
    print("=" * 60)

    valid_mask = result_df['error'].isna() | (result_df['error'] == 'No gold labels')
    valid_df = result_df[valid_mask]

    if len(valid_df) == 0:
        print("No valid results to summarize.")
        return

    total = len(result_df)
    valid_count = len(valid_df)
    error_count = total - valid_count

    print(f"\nTotal samples:   {total}")
    print(f"Valid samples:   {valid_count}")
    print(f"Errors:          {error_count}")

    print(f"\nAverage Metrics:")
    print(f"  Precision:           {valid_df['precision'].mean():.4f}")
    print(f"  Recall:              {valid_df['recall'].mean():.4f}")
    print(f"  F1 Score:            {valid_df['f1'].mean():.4f}")
    print(f"  Jaccard Exact:       {valid_df['jaccard_exact'].mean():.4f}")
    print(f"  Jaccard Relaxed:     {valid_df['jaccard_relaxed'].mean():.4f}")

    print(f"\nClause Statistics:")
    print(f"  Avg predicted clauses: {valid_df['num_pred'].mean():.2f}")
    print(f"  Avg gold clauses:      {valid_df['num_gold'].mean():.2f}")

    total_gold = valid_df['num_gold'].sum()
    total_pred = valid_df['num_pred'].sum()
    if total_gold > 0:
        loss_ratio = (total_gold - total_pred) / total_gold
        print(f"  Position recovery loss: {loss_ratio:.1%}")

    print("=" * 60)

# IsaNLP RST Parser

In [6]:
from isanlp_rst.parser import Parser

version = 'gumrrg'

parser = Parser(hf_model_name='tchewik/isanlp_rst_v3',
                hf_model_version=version,
                cuda_device=0)

text = """
Я помню только два раза за всю жизнь, когда я позволила себе заплакать не в одиночестве, а теперь эти грёбаные взрывы очень тяжело сдерживать
"""

res = parser(text)

print(vars(res['rst'][0]))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


best_weights.pt:   0%|          | 0.00/2.49G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

relation_table.txt:   0%|          | 0.00/350 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

{'id': 5, 'left': (id=4, start=1, end=89), 'right': (id=2, start=90, end=142), 'relation': 'adversative', 'nuclearity': 'NN', 'proba': None, 'entropy': 0.00197, 'start': 1, 'end': 142, 'text': 'Я помню только два раза за всю жизнь, когда я позволила себе заплакать не в одиночестве, а теперь эти грёбаные взрывы очень тяжело сдерживать', '_exporter': None}


In [7]:
from isanlp_rst.utils import tree_to_dict
import json

result = parser(text)
serialised = tree_to_dict(result['rst'][0])
serialised

{'id': 5,
 'relation': 'adversative',
 'nuclearity': 'NN',
 'start': 1,
 'end': 142,
 'text': 'Я помню только два раза за всю жизнь, когда я позволила себе заплакать не в одиночестве, а теперь эти грёбаные взрывы очень тяжело сдерживать',
 'left': {'id': 4,
  'relation': 'elaboration',
  'nuclearity': 'NS',
  'start': 1,
  'end': 89,
  'text': 'Я помню только два раза за всю жизнь, когда я позволила себе заплакать не в одиночестве,',
  'left': {'id': 0,
   'relation': 'elementary',
   'nuclearity': '',
   'start': 1,
   'end': 38,
   'text': 'Я помню только два раза за всю жизнь,'},
  'right': {'id': 1,
   'relation': 'elementary',
   'nuclearity': '',
   'start': 39,
   'end': 89,
   'text': 'когда я позволила себе заплакать не в одиночестве,'}},
 'right': {'id': 2,
  'relation': 'elementary',
  'nuclearity': '',
  'start': 90,
  'end': 142,
  'text': 'а теперь эти грёбаные взрывы очень тяжело сдерживать'}}

In [9]:
def extract_clauses_from_dict(node: dict) -> list[tuple[int, int, str]]:
    clauses = []

    if node.get('relation') == 'elementary' or ('left' not in node and 'right' not in node):
        start = node['start']
        end = node['end']
        text = node.get('text', '').strip()
        if text:
            clauses.append((start, end, text))
    else:
        if 'left' in node and node['left']:
            clauses.extend(extract_clauses_from_dict(node['left']))
        if 'right' in node and node['right']:
            clauses.extend(extract_clauses_from_dict(node['right']))

    return clauses

def parse_text_with_isanlp(text: str, parser: Parser) -> list[tuple[int, int, str]]:
    try:
        result = parser(text)
        rst_tree = result['rst']

        if not rst_tree:
            return []

        tree_dict = tree_to_dict(rst_tree[0])
        clauses = extract_clauses_from_dict(tree_dict)
        clauses.sort(key=lambda x: x[0])

        return clauses
    except Exception as e:
        print(f"Error parsing text with IsaNLP: {e}")
        return []


def evaluate_dataset_isanlp(df, model_version: str = 'gumrrg', threshold: float = 0.85) -> pd.DataFrame:
    print("Initializing IsaNLP RST Parser...")
    parser = Parser(
        hf_model_name='tchewik/isanlp_rst_v3',
        hf_model_version=model_version,
        cuda_device=-1
    )
    print("Parser initialized. Starting evaluation...\n")

    results = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing samples"):
        try:
            pred_clauses = parse_text_with_isanlp(row['text'], parser)

            metrics = compute_metrics_and_predictions(
                row['text'],
                row['label'],
                pred_clauses,
                threshold
            )

            result_row = {
                "text": row['text'],
                "label": row['label'],
                **metrics
            }
            results.append(result_row)

        except Exception as e:
            print(f"Error on sample {idx}: {e}")
            results.append({
                "text": row['text'],
                "label": row['label'],
                "parser": "[]",
                "precision": 0.0,
                "recall": 0.0,
                "f1": 0.0,
                "jaccard_exact": 0.0,
                "jaccard_relaxed": 0.0,
                "num_pred": 0,
                "num_gold": 0,
                "error": str(e)
            })

    result_df = pd.DataFrame(results)

    columns_order = [
        "text", "label", "parser",
        "precision", "recall", "f1",
        "jaccard_exact", "jaccard_relaxed",
        "num_pred", "num_gold", "error"
    ]

    existing_columns = [col for col in columns_order if col in result_df.columns]
    result_df = result_df[existing_columns]

    return result_df

In [10]:
result_df = evaluate_dataset_isanlp(df, model_version='gumrrg', threshold=0.85)

Initializing IsaNLP RST Parser...
Parser initialized. Starting evaluation...



Processing samples: 100%|██████████| 200/200 [02:36<00:00,  1.28it/s]


In [11]:
result_df.head()

,text,label,parser,precision,recall,f1,jaccard_exact,jaccard_relaxed,num_pred,num_gold,error
0,"Я стала вновь жить, радоваться, с великолепным...","[{""start"":0,""end"":120,""text"":""Я стала вновь жи...","[{""start"": 0, ""end"": 31, ""text"": ""Я стала внов...",0.00,0.000000,0.0,0.000000,0.000000,5,1,None
1,появляются мысли о суициде или желание чтобы м...,"[{""start"":0,""end"":38,""text"":""появляются мысли ...","[{""start"": 0, ""end"": 38, ""text"": ""появляются м...",1.00,1.000000,1.0,1.000000,1.000000,5,5,None
2,"А ещё он пьет, это началось давно и продолжает...","[{""start"":0,""end"":13,""text"":""А ещё он пьет"",""l...","[{""start"": 0, ""end"": 14, ""text"": ""А ещё он пье...",0.75,0.857143,0.8,0.071429,0.666667,8,7,None
3,"и мне кажется теперь, что яникогда не поправлюсь.","[{""start"":0,""end"":20,""text"":""и мне кажется теп...","[{""start"": 0, ""end"": 21, ""text"": ""и мне кажетс...",1.00,1.000000,1.0,0.000000,1.000000,2,2,None
4,около недели назад моя биологическая мать с оп...,"[{""start"":0,""end"":88,""text"":""около недели наза...","[{""start"": 0, ""end"": 89, ""text"": ""около недели...",1.00,1.000000,1.0,0.000000,1.000000,2,2,None


In [12]:
print_summary_stats(result_df)


SUMMARY STATISTICS

Total samples:   200
Valid samples:   200
Errors:          0

Average Metrics:
  Precision:           0.7246
  Recall:              0.7789
  F1 Score:            0.7427
  Jaccard Exact:       0.0711
  Jaccard Relaxed:     0.6828

Clause Statistics:
  Avg predicted clauses: 4.24
  Avg gold clauses:      3.56
  Position recovery loss: -19.0%


# Clause Segmenter (A clause segmenting tool utilising Python's SpaCy.)

In [2]:
# ! python3 -m pip install clause-segmenter

In [7]:
from clause_segmenter import ClauseSegmenter

text = df.iloc[0]['text']
segmenter = ClauseSegmenter(pipeline='ru_core_news_lg')
clauses_ls = segmenter.get_clauses_as_list(text)
for clause in clauses_ls:
    print(clause)

✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Я стала вновь жить, радоваться
с великолепным аттестатом окончила школу
поступила туда, куда хотела, ходила в церковь


In [8]:
clauses_ls

['Я стала вновь жить, радоваться',
 'с великолепным аттестатом окончила школу',
 'поступила туда, куда хотела, ходила в церковь']

In [9]:
def _normalize_for_search(text: str) -> str:
    text = re.sub(r'\s+', ' ', text).strip().lower()
    text = re.sub(r'\s*([.,;:!?()\[\]{}""«»\-—])\s*', r'\1', text)
    return text

def _map_normalized_to_original(original: str, normalized: str, norm_pos: int) -> int:
    mapping = []
    orig_idx = 0
    norm_idx = 0

    while norm_idx < len(normalized) and orig_idx < len(original):
        if normalized[norm_idx] == original[orig_idx].lower():
            mapping.append(orig_idx)
            norm_idx += 1
        orig_idx += 1

    while len(mapping) < len(normalized):
        mapping.append(len(original) - 1)

    return mapping[norm_pos] if norm_pos < len(mapping) else 0

def get_clause_segmenter_positions(
    text: str,
    segmenter: ClauseSegmenter,
    similarity_threshold: float = 0.70
) -> List[Tuple[int, int, str]]:
    try:
        clauses_list = segmenter.get_clauses_as_list(text)
        if not clauses_list:
            return []

        result = []
        norm_original = _normalize_for_search(text)
        search_start_norm = 0

        for clause_text in clauses_list:
            norm_clause = _normalize_for_search(clause_text)

            idx = text.find(clause_text, search_start_norm)

            if idx != -1:
                start = idx
                end = idx + len(clause_text)
                result.append((start, end, text[start:end]))
                search_start_norm = len(_normalize_for_search(text[:end]))
                continue

            matcher = SequenceMatcher(None, norm_original[search_start_norm:], norm_clause)
            blocks = matcher.get_matching_blocks()

            best_match = None
            best_ratio = 0.0

            for match in blocks:
                a_idx, b_idx, size = match
                if size == 0:
                    continue

                ratio = size / len(norm_clause)

                if ratio > best_ratio and a_idx < 50:
                    best_ratio = ratio
                    best_match = (a_idx + search_start_norm, size)

            if best_match is not None and best_ratio >= similarity_threshold:
                norm_start, match_len = best_match

                orig_start = _map_normalized_to_original(text, norm_original, norm_start)

                orig_end_pos_in_norm = norm_start + match_len
                orig_end = _map_normalized_to_original(
                    text, norm_original, min(orig_end_pos_in_norm, len(norm_original)-1)
                )

                found_text = text[orig_start:orig_end].strip()

                if found_text:
                    result.append((orig_start, orig_end, found_text))
                    search_start_norm = orig_end_pos_in_norm
                else:
                    print(f"Warning: Empty match at norm_pos={norm_start}")
            else:
                print(f"Warning: Clause not found (ratio={best_ratio:.2f}): '{clause_text[:40]}...'")

        return result

    except Exception as e:
        print(f"Error in ClauseSegmenter processing: {e}")
        import traceback
        traceback.print_exc()
        return []


def evaluate_dataset_clause_segmenter(df, pipeline_name: str = 'ru_core_news_lg', threshold: float = 0.85) -> pd.DataFrame:
    print(f"Initializing ClauseSegmenter with pipeline='{pipeline_name}'...")
    segmenter = ClauseSegmenter(pipeline=pipeline_name)
    print("Segmenter initialized. Starting evaluation...\n")

    results = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing samples"):
        try:
            pred_clauses = get_clause_segmenter_positions(row['text'], segmenter)

            metrics = compute_metrics_and_predictions(
                row['text'],
                row['label'],
                pred_clauses,
                threshold
            )

            result_row = {"text": row['text'], "label": row['label'], **metrics}
            results.append(result_row)

        except Exception as e:
            print(f"Error on sample {idx}: {e}")
            results.append({
                "text": row['text'], "label": row['label'],
                "parser": "[]", "precision": 0.0, "recall": 0.0, "f1": 0.0,
                "jaccard_exact": 0.0, "jaccard_relaxed": 0.0,
                "num_pred": 0, "num_gold": 0, "error": str(e)
            })

    result_df = pd.DataFrame(results)
    columns_order = ["text", "label", "parser", "precision", "recall", "f1",
                     "jaccard_exact", "jaccard_relaxed", "num_pred", "num_gold", "error"]
    existing_columns = [col for col in columns_order if col in result_df.columns]

    return result_df[existing_columns]

In [10]:
result_df = evaluate_dataset_clause_segmenter(df, pipeline_name='ru_core_news_lg', threshold=0.85)

Initializing ClauseSegmenter with pipeline='ru_core_news_lg'...
Segmenter initialized. Starting evaluation...



Processing samples:   6%|▌         | 12/200 [00:00<00:09, 19.32it/s]

Processing samples:   8%|▊         | 17/200 [00:01<00:13, 13.52it/s]

Processing samples:  14%|█▍        | 29/200 [00:01<00:06, 27.09it/s]

Processing samples:  20%|█▉        | 39/200 [00:01<00:05, 31.13it/s]

...

Processing samples:  76%|███████▌  | 151/200 [00:03<00:00, 50.05it/s]

Processing samples:  83%|████████▎ | 166/200 [00:04<00:00, 35.43it/s]

Processing samples:  87%|████████▋ | 174/200 [00:04<00:00, 33.23it/s]

Processing samples:  92%|█████████▏| 184/200 [00:05<00:00, 37.50it/s]

Processing samples: 100%|██████████| 200/200 [00:05<00:00, 37.88it/s]


In [12]:
result_df.head()

,text,label,parser,precision,recall,f1,jaccard_exact,jaccard_relaxed,num_pred,num_gold,error
0,"Я стала вновь жить, радоваться, с великолепным...","[{""start"":0,""end"":120,""text"":""Я стала вновь жи...","[{""start"": 0, ""end"": 30, ""text"": ""Я стала внов...",0.000000,0.000000,0.000000,0.000000,0.000000,3,1,None
1,появляются мысли о суициде или желание чтобы м...,"[{""start"":0,""end"":38,""text"":""появляются мысли ...","[{""start"": 0, ""end"": 38, ""text"": ""появляются м...",0.666667,0.400000,0.500000,0.142857,0.333333,3,5,None
2,"А ещё он пьет, это началось давно и продолжает...","[{""start"":0,""end"":13,""text"":""А ещё он пьет"",""l...","[{""start"": 0, ""end"": 13, ""text"": ""А ещё он пье...",0.500000,0.428571,0.461538,0.300000,0.300000,6,7,None
3,"и мне кажется теперь, что яникогда не поправлюсь.","[{""start"":0,""end"":20,""text"":""и мне кажется теп...","[{""start"": 0, ""end"": 20, ""text"": ""и мне кажетс...",1.000000,1.000000,1.000000,1.000000,1.000000,2,2,None
4,около недели назад моя биологическая мать с оп...,"[{""start"":0,""end"":88,""text"":""около недели наза...","[{""start"": 0, ""end"": 112, ""text"": ""около недел...",1.000000,0.500000,0.666667,0.000000,0.500000,1,2,None


In [13]:
print_summary_stats(result_df)


SUMMARY STATISTICS

Total samples:   200
Valid samples:   200
Errors:          0

Average Metrics:
  Precision:           0.4562
  Recall:              0.4213
  F1 Score:            0.4298
  Jaccard Exact:       0.3355
  Jaccard Relaxed:     0.3832

Clause Statistics:
  Avg predicted clauses: 2.10
  Avg gold clauses:      3.56
  Position recovery loss: 41.2%
